In [1]:
from src.collectors.ccnews_extractor import (
    domain_allowed,
    url_is_article,
    load_processed,
    save_processed,
    load_month_paths,
    process_warc_stream,
    run_one_warc,
    perf_counter,
    Path,
    os,
    defaultdict
)

In [2]:
BASE_URL = "https://data.commoncrawl.org/"

MONTHS = [
    "2025/01","2025/02","2025/03","2025/04","2025/05","2025/06",
    "2025/07","2025/08","2025/09","2025/10","2025/11","2025/12",
    "2026/01","2026/02","2026/03"
]

parent_dir = Path(os.curdir).resolve().parent
DATA_DIR = os.path.join(parent_dir, "data/raw/ccnews")
os.makedirs(DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(DATA_DIR, "articles_filtered.jsonl")
PROCESSED_LOG = os.path.join(DATA_DIR, "processed_warcs_filtered.json")

MAX_HTML_SIZE = 1_000_000
WRITE_BATCH_SIZE = 200
MAX_ITEMS_PER_WARC = 100000
downsample_rate = None

ALLOWED_DOMAINS = {
    "reuters.com",
    "bbc.com",
    "bbc.co.uk",
    "ft.com",
    "bloomberg.com",
    "cnbc.com",
    "wsj.com",
    "nytimes.com",
    "economist.com",
    "theguardian.com",
    "washingtonpost.com",
    "apnews.com",
    "businessinsider.com",
    "marketwatch.com",
    "yahoo.com",
    "forbes.com",
    "cnn.com",
    "nbcnews.com",
    "abcnews.go.com",
    "aljazeera.com",

    # list of en domains
    'www.channelnewsasia.com',
    'www.thenationalnews.com',
    # 'www.swissinfo.ch',
    'www.straitstimes.com',
    'vietnamnews.vn',
    'abcnews.go.com',
    'www.cbsnews.com',
    'www.foxnews.com',
    'www.latimes.com',
    'globalnews.ca',
    'www.ctvnews.ca',
    'www.telegraph.co.uk',
    'www.independent.co.uk',
    'www.the-independent.com',
    'www.irishtimes.com',
    'www.scotsman.com',
    'www.yorkshirepost.co.uk',
    # 'timesofindia.indiatimes.com',
    # 'economictimes.indiatimes.com',
    'indianexpress.com',
    'www.ndtv.com',
    # 'www.business-standard.com',
    # 'www.livemint.com',
    # 'www.businesstoday.in',
    'gulfnews.com',
    'www.khaleejtimes.com',
    'allafrica.com',
    'www.timeslive.co.za',
    'www.businesslive.co.za',
    'thewest.com.au',
    'www.nzherald.co.nz',
    # 'www.nasdaq.com',
    # 'seekingalpha.com',
    'qz.com',
    # 'www.barchart.com',
    # 'www.pymnts.com',

    # ru domains
    'www.tass.ru',
    # 'ria.ru', # bad format of html
    'www.interfax.ru',
    'www.rbc.ru',
    'www.vedomosti.ru',
    'www.kommersant.ru',
    'www.lenta.ru',
    'www.nur.kz',
    'www.zakon.kz'
}

BAD_PATHS = [
    "/video/",
    "/videos/",
    "/live/",
    "/gallery/",
    "/podcast/",
    'sport',
    'crimea'
]

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
from collections import defaultdict
from time import perf_counter
import random
from src.collectors.ccnews_extractor import run_one_warc

cfg = {
    "bad_paths": BAD_PATHS,
    "max_html_size": MAX_HTML_SIZE,
    "write_batch_size": WRITE_BATCH_SIZE,
    "max_items_per_warc": MAX_ITEMS_PER_WARC,
    "downsample_rate": downsample_rate,
    "allowed_domains": ALLOWED_DOMAINS,
    "allowed_langs": ("en", "ru"),
    "output_root": DATA_DIR,
}

MAX_WORKERS = 4
t0 = perf_counter()

processed = load_processed(PROCESSED_LOG)
month_paths = {m: load_month_paths(BASE_URL, m) for m in MONTHS}

warc_rels = []
for m in MONTHS:
    warc_rels.extend(month_paths[m])
random.shuffle(warc_rels)

warc_rels = [w for w in warc_rels if w not in processed]

total = defaultdict(float)

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    future_map = {
        ex.submit(run_one_warc, BASE_URL + warc_rel, cfg): warc_rel
        for warc_rel in warc_rels
    }

    for fut in as_completed(future_map):
        warc_rel = future_map[fut]
        try:
            res = fut.result()
            stats = res["stats"]

            for k, v in stats.items():
                total[k] += v

            processed.add(warc_rel)
            save_processed(processed, PROCESSED_LOG)

            print(
                "WARC stats | "
                f"warc={warc_rel} "
                f"written={int(stats.get('written', 0))} "
                f"warc_s={stats.get('warc_seconds', 0):.2f} "
                f"out={res.get('output_path')}"
            )
        except Exception as e:
            print(f"FAILED {warc_rel}: {e}")

elapsed = perf_counter() - t0
print(
    "Run summary | "
    f"written={int(total.get('written', 0))} "
    f"records={int(total.get('records_seen', 0))} "
    f"extract_s={total.get('extract_seconds', 0):.2f} "
    f"total_s={elapsed:.2f}"
)

FAILED crawl-data/CC-NEWS/2025/02/CC-NEWS-20250217163407-00778.warc.gz: ('Connection broken: IncompleteRead(192936887 bytes read, 879803790 more expected)', IncompleteRead(192936887 bytes read, 879803790 more expected))
WARC stats | warc=crawl-data/CC-NEWS/2026/02/CC-NEWS-20260208103549-06729.warc.gz written=199 warc_s=156.72 out=/Users/nikitakuznecov/Desktop/petprojects/market_narratives_engine/data/raw/ccnews/2026/02/CC-NEWS-20260208103549-06729.jsonl
WARC stats | warc=crawl-data/CC-NEWS/2025/01/CC-NEWS-20250131141004-00552.warc.gz written=317 warc_s=165.18 out=/Users/nikitakuznecov/Desktop/petprojects/market_narratives_engine/data/raw/ccnews/2025/01/CC-NEWS-20250131141004-00552.jsonl
WARC stats | warc=crawl-data/CC-NEWS/2025/10/CC-NEWS-20251028011855-04864.warc.gz written=219 warc_s=179.32 out=/Users/nikitakuznecov/Desktop/petprojects/market_narratives_engine/data/raw/ccnews/2025/10/CC-NEWS-20251028011855-04864.jsonl
WARC stats | warc=crawl-data/CC-NEWS/2025/06/CC-NEWS-2025060820554

- перезапустить парсинг с новыми доменами
- сделать партицирование по месяцам в parquet файлы
- для очистки данных - посмотреть % статей по доменам с высоким уровнем повторов словосочетаний - фэйлы с парсингом html
